# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [5]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [43]:
MODEL = "gpt-4.1-mini"
# BASE_URL = "http://localhost:11434/v1"
url="https://github.com/mwarsi2784"
api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI()

In [38]:
links = fetch_website_links("https://github.com/mwarsi2784")

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [23]:
get_link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [24]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [39]:
get_links_user_prompt("https://github.com/mwarsi2784")

'\nHere is the list of links on the website https://github.com/mwarsi2784 -\nPlease decide which of these are relevant web links for a brochure about the company, \nrespond with the full https URL in JSON format.\nDo not include Terms of Service, Privacy, email links.\n\nLinks (some might be relative links):\n\n#start-of-content\n/\n/login?return_to=https%3A%2F%2Fgithub.com%2Fmwarsi2784\nhttps://github.com/features/copilot\nhttps://github.com/features/ai/github-app\nhttps://github.com/mcp\nhttps://github.com/features/actions\nhttps://github.com/features/codespaces\nhttps://github.com/features/issues\nhttps://github.com/features/code-review\nhttps://github.com/security/advanced-security\nhttps://github.com/security/advanced-security/code-security\nhttps://github.com/security/advanced-security/secret-protection\nhttps://github.com/why-github\nhttps://docs.github.com\nhttps://github.blog\nhttps://github.blog/changelog\nhttps://github.com/marketplace\nhttps://github.com/features\nhttps://g

In [40]:
def select_relevant_links(url):
    respone = openai.chat.completions.create(model=MODEL,messages=[
        {"role":"system", "content":get_link_system_prompt},
        {"role":"user", "content":get_links_user_prompt(url)}
    ], response_format={"type":"json_object"})
    result = respone.choices[0].message.content
    links = json.loads(result)
    return links

In [ ]:
select_relevant_links("https://github.com/mwarsi2784")

{'links': [{'type': 'homepage', 'url': 'https://github.com'},
  {'type': 'about page', 'url': 'https://github.com/why-github'},
  {'type': 'features overview', 'url': 'https://github.com/features'},
  {'type': 'enterprise page', 'url': 'https://github.com/enterprise'},
  {'type': 'enterprise support',
   'url': 'https://github.com/enterprise/premium-support'},
  {'type': 'team page', 'url': 'https://github.com/team'},
  {'type': 'pricing page', 'url': 'https://github.com/pricing'},
  {'type': 'customer stories / case studies',
   'url': 'https://github.com/customer-stories'},
  {'type': 'solutions', 'url': 'https://github.com/solutions'},
  {'type': 'partners', 'url': 'https://github.com/partners'},
  {'type': 'resources', 'url': 'https://github.com/resources'},
  {'type': 'contact page', 'url': 'https://github.com/contact'}]}

In [28]:
def select_relevant_links(url):
    respone = openai.chat.completions.create(model=MODEL,messages=[
        {"role":"system", "content":get_link_system_prompt},
        {"role":"user", "content":get_links_user_prompt(url)}
    ], response_format={"type":"json_object"})
    result = respone.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [29]:
select_relevant_links(url)

Found 14 relevant links


{'links': [{'type': 'homepage', 'url': 'https://github.com'},
  {'type': 'about page', 'url': 'https://github.com/why-github'},
  {'type': 'blog', 'url': 'https://github.blog'},
  {'type': 'enterprise page', 'url': 'https://github.com/enterprise'},
  {'type': 'pricing page', 'url': 'https://github.com/pricing'},
  {'type': 'features page', 'url': 'https://github.com/features'},
  {'type': 'team page', 'url': 'https://github.com/team'},
  {'type': 'security / trust', 'url': 'https://github.com/trust-center'},
  {'type': 'security product',
   'url': 'https://github.com/security/advanced-security'},
  {'type': 'customer stories', 'url': 'https://github.com/customer-stories'},
  {'type': 'partners', 'url': 'https://github.com/partners'},
  {'type': 'marketplace', 'url': 'https://github.com/marketplace'},
  {'type': 'resources', 'url': 'https://github.com/resources'},
  {'type': 'contact page', 'url': 'https://github.com/contact'}]}

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [34]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n Relevant Links:\n"
    for link in relevant_links['links']:
        result+=f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [35]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Found 13 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
prism-ml/Ternary-Bonsai-27B-gguf
Updated
about 20 hours ago
•
23
•
383
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
1 day ago
•
2.01M
•
2.2k
zai-org/GLM-5.2
Updated
13 days ago
•
490k
•
3.98k
prism-ml/Bonsai-27B-gguf
Updated
about 19 hours ago
•
513
•
231
bottlecapai/ThinkingCap-Qwen3.6-27B
Updated
5 days ago
•
6.21k
•
362
Browse 2M+ models
Spaces
Running
on
Zero
Agents
60

In [41]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [42]:
def get_broucher_user_prompt(company_name,url):
    user_prompt = f""""
    You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt+=fetch_page_and_all_relevant_links(url)
    user_prompt=user_prompt[:3000]
    return user_prompt

In [ ]:
get_broucher_user_prompt("HuggingFace","https://huggingface.co")

In [49]:
def create_brochure(company_name,url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system", "content":brochure_system_prompt},
            {"role":"user", "content":get_broucher_user_prompt(company_name,url)}
        ]
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [50]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## Who We Are

**Hugging Face** is the AI community building the future of machine learning. We provide the collaborative platform where the global machine learning community creates, discovers, and shares models, datasets, and applications. Our vision is to empower developers, researchers, and enterprises by making AI technology accessible and usable.

---

## Our Platform

- **Models:** Access and browse over 2 million machine learning models, continuously updated and contributed by a thriving community.
- **Datasets:** Explore more than 500,000 datasets for diverse applications, supporting innovation across industries.
- **Spaces:** Host and discover interactive machine learning apps in our community-run spaces.
- **Buckets:** Secure and scalable storage solutions for models and datasets.

Our platform supports unlimited public hosting and seamless collaboration, fostering an open environment where AI advancements flourish.

---

## Our Community

Hugging Face is more than a platform—it's a vibrant community of over 100,000 AI enthusiasts ranging from scholars publishing cutting-edge research papers, to developers contributing new datasets or refining models. Engagement happens every day on our:

- **Discord channel**
- **Community Forums**
- **GitHub repositories**

We celebrate openness, continuous learning, and shared success—a place where all contributions help shape the future of AI.

---

## Enterprise Solutions

For businesses seeking to leverage AI at scale, we offer:

- **Hugging Face PRO:** Advanced enterprise features for enhanced productivity and support.
- **Enterprise Support:** Dedicated technical assistance to ensure your AI deployments succeed.
- **Inference Providers and Endpoints:** Scalable and efficient infrastructure to run your models.
- **Storage Buckets:** Robust, secure storage for your critical AI assets.

Our enterprise offerings are designed to integrate smoothly with existing workflows and meet rigorous industry standards.

---

## Careers and Culture

At Hugging Face, we believe in innovation powered by collaboration and diversity. Our culture is rooted in openness, curiosity, and shared growth, fostering an environment where everyone—from newcomers to veterans—can experiment, learn, and contribute.

We continuously seek passionate engineers, researchers, community managers, and AI enthusiasts eager to make an impact in a fast-moving field. Here, your ideas and contributions directly shape the ML landscape of tomorrow.

---

## Join Us

Whether you are a developer, researcher, enterprise leader, or learner, Hugging Face offers tools and a community to accelerate your AI journey:

- **Explore AI apps and models**
- **Contribute or find datasets**
- **Collaborate with peers worldwide**
- **Join the conversation on Discord and forums**

Discover more and get involved at [huggingface.co](https://huggingface.co)

---

**Hugging Face**  
The AI community building the future.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [53]:
def stream_brochure(company_name,url):
    stream = openai.chat.completions.create(
        model=MODEL,
                messages=[
            {"role":"system", "content":brochure_system_prompt},
            {"role":"user", "content":get_broucher_user_prompt(company_name,url)}
        ],
        stream=True
    )
    response =""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response+=chunk.choices[0].delta.content or ''"week1 EXERCISE.ipynb"
        update_display(Markdown(response),display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [54]:
stream_brochure("HuggingFace", "https://huggingface.co")

week1 EXERCISE.ipynb# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a pioneering AI company and vibrant community dedicated to building the future of artificial intelligence. Serving as a collaborative platform, it enables machine learning practitioners, researchers, and developers worldwide to create, discover, and share machine learning models, datasets, and applications with ease.

---

## What We Offer

- **Models:** Access and contribute to a vast library of over 2 million machine learning models, ranging from natural language processing to computer vision and beyond. The models are frequently updated and cover diverse research areas and practical applications.

- **Datasets:** Explore and share from a collection of over 500,000 curated datasets to train and evaluate machine learning models.

- **Spaces:** Deploy and experiment with over 1 million AI applications directly on the platform, enabling seamless interaction with models in real-time.

- **Buckets:** Secure and scalable storage solutions for datasets and model files.

- **Enterprise Solutions:** Hugging Face supports businesses with enterprise-grade features, including professional support, inference endpoints, storage buckets, and collaboration tools tailored for teams.

---

## The Community & Culture

Hugging Face fosters a **collaborative and inclusive AI community** where openness and sharing are key. With over 100,000 followers and active participation on platforms like Discord, GitHub, and forums, the community thrives on continuous learning, sharing the latest research, and advancing machine intelligence together.

The culture encourages contributions from both industry and academia, welcoming developers, researchers, and enthusiasts to join in pushing the boundaries of AI innovation.

---

## Customers & Impact

Hugging Face serves a diverse group of users including:

- AI researchers and academic institutions leveraging state-of-the-art models and datasets for cutting-edge research.
- Machine learning engineers and developers building AI-powered applications.
- Enterprises seeking scalable and customizable AI solutions with dedicated support.
- AI enthusiasts and students accessing resources to learn and experiment.

---

## Careers at Hugging Face

Hugging Face is a dynamic and rapidly growing company actively seeking **talented professionals** passionate about AI and machine learning.

**Why join?**

- Work alongside leading experts in AI and open source.
- Contribute to impactful projects used globally by millions.
- Engage in a community-driven environment emphasizing innovation, collaboration, and transparency.
- Access opportunities to publish and share your work with a broad audience.

Visit the Hugging Face website and GitHub for latest openings and contribute directly to the AI community.

---

## Connect & Learn More

- Website: [huggingface.co](https://huggingface.co)
- Community Forums & Discord: Join active discussions and get support.
- GitHub: Explore code, models, and contribute to projects.
- Blog & Daily Papers: Stay updated with the latest AI research and trends.

---

**Hugging Face** is where the AI community collaborates today to build the intelligent solutions of tomorrow. Join us and be part of the future!week1 EXERCISE.ipynb

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>